<a href="https://colab.research.google.com/github/rakshitha2006gowda-art/Banking-FAQ-s-Assistant/blob/main/Day7_HandsOn_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **RAG**

**Install libraries**

In [1]:
!pip install -q -U openai chromadb langchain-text-splitters pypdf python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/

**import libraries**

In [2]:
import os
import openai
import chromadb

from google.colab import files, userdata

from pypdf import PdfReader
from docx import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

**upload study material**

In [4]:
uploaded = files.upload()

file_path = list(uploaded.keys())[0]

print("Uploaded File:", file_path)

Saving RAG_Implementation_AI_Smart_Learning_Assistant.pdf to RAG_Implementation_AI_Smart_Learning_Assistant.pdf
Uploaded File: RAG_Implementation_AI_Smart_Learning_Assistant.pdf


In [5]:
ext = os.path.splitext(file_path)[1].lower()

text = ""

if ext == ".pdf":
    reader = PdfReader(file_path)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

elif ext == ".txt":
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

elif ext == ".docx":
    doc = Document(file_path)

    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"

else:
    raise ValueError(
        f"Unsupported file type: {ext}. "
        "Please upload PDF, TXT, or DOCX."
    )

print("Document loaded successfully!")
print("Characters:", len(text))

print("\nFirst 1000 characters:")
print(text[:1000])

Document loaded successfully!
Characters: 11003

First 1000 characters:
AI Smart Learning Assistant – RAG Implementation | Page 1
 RAG Implementation for an
AI Smart Learning Assistant
 Technical Implementation Document
Retrieval-Augmented Generation (RAG) based Educational Assistant
Project Area
Artificial Intelligence / Generative AI / NLP
Core Technique
Retrieval-Augmented Generation (RAG)
Application
AI Smart Learning Assistant
Primary Goal
Provide grounded, context-aware answers from trusted learning materials
Suggested Stack
Python, OpenAI API, embeddings, vector database, Streamlit/FastAPI

AI Smart Learning Assistant – RAG Implementation | Page 2
1. Introduction
An AI Smart Learning Assistant is an educational application that helps students understand subjects, search learning
resources, ask questions, generate explanations, and revise concepts. A standard language model can answer general
questions, but it may not know the student's uploaded notes, syllabus, textbooks, lectur

**chunking**

In [6]:
from langchain_core.documents import Document as LCDocument

document = LCDocument(page_content=text)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents([document])

print("Total Chunks:", len(docs))

Total Chunks: 29


In [11]:
from google.colab import userdata
api_key=userdata.get('api_key')
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://nexusapi.navigatelabs.ai"
)

In [12]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="rag"
)

**vector database**

In [8]:
chroma_client = chromadb.Client()

# Delete old collection if it exists
try:
    chroma_client.delete_collection("ai_smart_learning")
except:
    pass

collection = chroma_client.create_collection(
    name="ai_smart_learning"
)

print("ChromaDB collection created!")

ChromaDB collection created!


**embeddings**

In [18]:
try:
    chroma_client.delete_collection("rag")
except:
    pass

collection = chroma_client.create_collection(
    name="rag"
)

for i, doc in enumerate(docs):

    text = doc.page_content

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("Embeddings stored successfully!")

Embeddings stored successfully!


**checking the vector database**

In [14]:
print("Documents stored in ChromaDB:", collection.count())

Documents stored in ChromaDB: 20


**ask a question**

In [15]:
query = input("Ask your study question: ")

print("\nQuestion:", query)

Ask your study question: what is rag?

Question: what is rag?


**question to embedding**

In [16]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_embedding = query_response.data[0].embedding

**similarity search**

In [17]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

retrieved_chunks = results["documents"][0]

print("Retrieved Chunks:")
print("=" * 60)

for i, chunk in enumerate(retrieved_chunks):

    print(f"\nChunk {i+1}")
    print("-" * 60)
    print(chunk)

Retrieved Chunks:

Chunk 1
------------------------------------------------------------

Support personalized learning such as explanations, summaries, and quizzes.
4. What is RAG?
Retrieval-Augmented Generation is an architecture in which information retrieval is combined with a generative
language model. Instead of sending only the student's question to the model, the system retrieves relevant passages
from a knowledge base and places those passages in the model's context.
Basic RAG flow: Documents → Chunking → Embeddings → Vector Database → Similarity Search → Relevant

Chunk 2
------------------------------------------------------------
AI Smart Learning Assistant – RAG Implementation | Page 1
 RAG Implementation for an
AI Smart Learning Assistant
 Technical Implementation Document
Retrieval-Augmented Generation (RAG) based Educational Assistant
Project Area
Artificial Intelligence / Generative AI / NLP
Core Technique
Retrieval-Augmented Generation (RAG)
Application
AI Smart Learn

**retrieved context**

In [19]:
context = "\n\n".join(retrieved_chunks)

print("\nRetrieved context created successfully!")


Retrieved context created successfully!


In [20]:
prompt = f"""
You are an AI Smart Learning Assistant.

Answer the student's question using the study material provided
in the context below.

Instructions:
1. Give a clear and accurate answer.
2. Use simple student-friendly language.
3. Explain important concepts step by step when needed.
4. Use examples when useful.
5. Do not invent information that is not supported by the context.
6. If the answer is not available in the study material, say:
   "This information is not available in the uploaded study material."
7. Keep the answer focused on the student's question.

Study Material:
{context}

Student Question:
{query}
"""

In [21]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI Smart Learning Assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

answer = response.choices[0].message.content

print("\nFINAL ANSWER")
print("=" * 60)
print(answer)


FINAL ANSWER
RAG, or Retrieval-Augmented Generation, is a method used in AI to improve how it answers questions. It combines two things: retrieving relevant information from a knowledge base (like textbooks or notes) and generating a clear answer using a language model. 

Here's how it works step by step:
1. When you ask a question, the system first searches for the most relevant passages from the stored educational materials.
2. These relevant passages are then used as context.
3. The language model uses this context to generate a grounded, accurate answer.

This way, the AI can give more reliable and context-aware responses based on trusted learning materials.
